# LeetCode 2054 — Two Best Non-Overlapping Events bằng Python

Link đề: https://leetcode.com/problems/two-best-non-overlapping-events/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Dynamic Programming, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maxTwoEvents(self, events: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài "chọn **tối đa hai** khoảng không giao nhau với tổng giá trị lớn nhất" thành: với mỗi sự kiện làm sự kiện **sau**, ghép với sự kiện **trước** tốt nhất đã kết thúc trước khi nó bắt đầu.
2. Dùng **sắp xếp + mảng max hậu tố + `bisect`** để trả lời "giá trị lớn nhất trong các sự kiện bắt đầu từ thời điểm `t` trở đi" trong `O(log n)`.
3. Dùng **quét theo thời gian + heap** để giữ "giá trị lớn nhất trong các sự kiện đã kết thúc".
4. Xử lý chính xác **biên bao gồm** (inclusive): sự kiện sau phải bắt đầu ở `≥ end + 1`.
5. Không quên lựa chọn **chỉ một** sự kiện.

## 2. Đề bài & ví dụ

Cho mảng `events`, `events[i] = [start_i, end_i, value_i]`: sự kiện `i` diễn ra từ `start_i` tới `end_i` (**bao gồm** cả hai đầu), tham dự được giá trị `value_i`. Chọn **tối đa hai** sự kiện **không chồng nhau** để tổng giá trị lớn nhất. Trả về tổng đó.

Vì thời gian bao gồm hai đầu: nếu tham dự sự kiện kết thúc lúc `t`, sự kiện tiếp theo phải bắt đầu **từ `t + 1`** trở đi.

**Ví dụ 1:** `[[1,3,2],[4,5,2],[2,4,3]]` → `4` (sự kiện 0 và 1).

**Ví dụ 2:** `[[1,3,2],[4,5,2],[1,5,5]]` → `5` (chỉ chọn sự kiện 2).

**Ví dụ 3:** `[[1,5,3],[1,5,1],[6,6,5]]` → `8` (sự kiện 0 và 2).

**Ràng buộc:** `2 ≤ n ≤ 10⁵`; `1 ≤ start ≤ end ≤ 10⁹`; `1 ≤ value ≤ 10⁶`.

In [1]:
import random
import time
import bisect
import heapq
from typing import List

examples = [([[1, 3, 2], [4, 5, 2], [2, 4, 3]], 4), ([[1, 3, 2], [4, 5, 2], [1, 5, 5]], 5), ([[1, 5, 3], [1, 5, 1], [6, 6, 5]], 8)]
for ev, exp in examples:
    print(f"events={ev} -> kỳ vọng {exp}")

events=[[1, 3, 2], [4, 5, 2], [2, 4, 3]] -> kỳ vọng 4
events=[[1, 3, 2], [4, 5, 2], [1, 5, 5]] -> kỳ vọng 5
events=[[1, 5, 3], [1, 5, 1], [6, 6, 5]] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Hai khoảng đóng không giao nhau

`[a, b]` và `[c, d]` (đóng) không giao nhau ⇔ `b < c` hoặc `d < a`. Với số nguyên, `b < c` ⇔ `c ≥ b + 1`.

In [2]:
def disjoint(e, f):
    return e[1] < f[0] or f[1] < e[0]

print(disjoint([1, 3, 2], [4, 5, 2]), disjoint([1, 3, 2], [3, 5, 2]), "(chung thời điểm 3 -> giao nhau)")

True False (chung thời điểm 3 -> giao nhau)


### 3.2 Mảng max hậu tố

`suf[i] = max(A[i], A[i+1], …, A[n−1])`, tính từ phải sang trái trong `O(n)`. Trả lời "max của mọi phần tử từ vị trí `i` trở đi" trong `O(1)`.

### 3.3 `bisect_left` trên danh sách đã sắp

Sắp các sự kiện theo `start`; vị trí đầu tiên có `start ≥ t` là `bisect_left(starts, t)`.

In [3]:
starts = [1, 2, 4, 4, 7]
for t in (0, 4, 5, 8):
    print(f"start >= {t}: từ chỉ số {bisect.bisect_left(starts, t)}")

start >= 0: từ chỉ số 0
start >= 4: từ chỉ số 2
start >= 5: từ chỉ số 4
start >= 8: từ chỉ số 5


### 3.4 Heap (hàng đợi ưu tiên)

`heapq` là min-heap: lấy phần tử nhỏ nhất `O(log n)`. Dùng để lấy ra lần lượt các sự kiện theo **thời điểm kết thúc** tăng dần trong lúc quét.

### 3.5 Thời gian tới `10⁹`: không lập mảng theo thời gian

Không thể tạo mảng `best[t]` cho mọi thời điểm `t ≤ 10⁹`. Ta chỉ làm việc với **các mốc xuất hiện trong dữ liệu** (tối đa `2n`): sắp xếp chúng, dùng `bisect` để nhảy tới mốc cần. Nếu muốn mảng, có thể **nén tọa độ** (gán chỉ số cho mỗi mốc khác nhau) — kết quả tương đương.

In [4]:
ev = examples[0][0]
coords = sorted({x for s, e, _ in ev for x in (s, e + 1)})
idx = {c: i for i, c in enumerate(coords)}
print("các mốc:", coords, "| nén:", idx)

các mốc: [1, 2, 4, 5, 6] | nén: {1: 0, 2: 1, 4: 2, 5: 3, 6: 4}


## 4. Ý tưởng nền tảng / trực giác

Mọi lời giải là một trong hai dạng: **một** sự kiện (giá trị lớn nhất), hoặc **hai** sự kiện `e` (trước) và `f` (sau) với `f.start ≥ e.end + 1`. Cố định `e` làm sự kiện **trước**: cần `max value` trong các sự kiện có `start ≥ e.end + 1`. Nếu sắp theo `start`, tập đó là một **hậu tố** ⇒ mảng max hậu tố.

Tính tay **Ví dụ 3** (sắp theo start: `[1,5,3], [1,5,1], [6,6,5]`, `suf = [5, 5, 5]`):

| e (trước) | cần start ≥ | chỉ số hậu tố | max hậu tố | tổng |
|---|---|---|---|---|
| `[1,5,3]` | 6 | 2 | 5 | **8** |
| `[1,5,1]` | 6 | 2 | 5 | 6 |
| `[6,6,5]` | 7 | 3 (rỗng) | 0 | 5 |

In [5]:
ev = sorted(examples[2][0])
st = [e[0] for e in ev]; suf = [0] * (len(ev) + 1)
for i in range(len(ev) - 1, -1, -1): suf[i] = max(suf[i + 1], ev[i][2])
for e in ev:
    j = bisect.bisect_left(st, e[1] + 1)
    print(f"e={e}: start >= {e[1] + 1} từ chỉ số {j}, max hậu tố {suf[j]}, tổng {e[2] + suf[j]}")

e=[1, 5, 1]: start >= 6 từ chỉ số 2, max hậu tố 5, tổng 6
e=[1, 5, 3]: start >= 6 từ chỉ số 2, max hậu tố 5, tổng 8
e=[6, 6, 5]: start >= 7 từ chỉ số 3, max hậu tố 0, tổng 5


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

In [6]:
def max_two_bruteforce(events):
    best = max(v for _, _, v in events)                   # chỉ một sự kiện
    n = len(events)
    for i in range(n):
        for j in range(i + 1, n):
            if disjoint(events[i], events[j]):
                best = max(best, events[i][2] + events[j][2])
    return best

for ev, exp in examples:
    got = max_two_bruteforce(ev); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
5 kỳ vọng 5
8 kỳ vọng 8


**Độ phức tạp:** `O(n²)` = `5·10⁹` cặp với `n = 10⁵` ⇒ TLE.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cố định sự kiện trước):** đáp án `= max(max value, max_e (e.value + best_after(e.end + 1)))` với `best_after(t)` = giá trị lớn nhất của sự kiện có `start ≥ t` (0 nếu không có).

**Insight 2 (hậu tố):** sắp theo `start` ⇒ `best_after(t) = suf[bisect_left(starts, t)]`. Tổng `O(n log n)`.

**Insight 3 (quét đối xứng):** hoặc cố định sự kiện **sau** `f`: cần max value của các sự kiện có `end < f.start`. Quét `f` theo `start` tăng dần, đẩy vào "kho đã kết thúc" mọi sự kiện có `end < f.start` (lấy ra từ heap theo `end`), giữ `best_ended`.

**Invariant (quét):** khi xét `f`, `best_ended` = max value của **mọi** sự kiện có `end < f.start`.

### 6.1 Minh họa: vì sao phải xét cả trường hợp một sự kiện

Nếu một sự kiện có giá trị rất lớn và giao với mọi sự kiện khác, đáp án chỉ là nó (Ví dụ 2). Code chỉ xét các **cặp** sẽ sai. In mọi cặp không giao nhau của Ví dụ 2:

In [7]:
ev = examples[1][0]
pairs = [(ev[i], ev[j], ev[i][2] + ev[j][2]) for i in range(3) for j in range(i + 1, 3) if disjoint(ev[i], ev[j])]
print("các cặp hợp lệ:", pairs, "| max cặp:", max(p[2] for p in pairs), "| max đơn:", max(e[2] for e in ev))

các cặp hợp lệ: [([1, 3, 2], [4, 5, 2], 4)] | max cặp: 4 | max đơn: 5


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp theo start + max hậu tố + `bisect` (bản nộp)

In [8]:
def max_two_suffix(events):
    ev = sorted(events)
    starts = [e[0] for e in ev]
    n = len(ev)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(suf[i + 1], ev[i][2])
    best = 0
    for s, e, v in ev:
        j = bisect.bisect_left(starts, e + 1)
        best = max(best, v + suf[j])                   # suf[n] = 0 bao gồm trường hợp một sự kiện
    return best

print([max_two_suffix(ev) for ev, _ in examples])

[4, 5, 8]


Mẹo `suf[n] = 0`: khi không có sự kiện nào bắt đầu sau `e`, tổng là `v + 0` — chính là trường hợp **một** sự kiện. Không cần xử lý riêng.

**Độ phức tạp Cách 2:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Quét theo start + heap theo end

In [9]:
def max_two_heap(events):
    ev = sorted(events)                                # theo start
    heap = []                                          # (end, value) các sự kiện đã xét
    best_ended = best = 0
    for s, e, v in ev:
        while heap and heap[0][0] < s:                 # kết thúc trước khi f bắt đầu
            best_ended = max(best_ended, heapq.heappop(heap)[1])
        best = max(best, v + best_ended)
        heapq.heappush(heap, (e, v))
    return best

print([max_two_heap(ev) for ev, _ in examples])

[4, 5, 8]


Một sự kiện đã bị lấy khỏi heap vì `end < s` thì cũng `end < s'` với mọi `s' ≥ s` sau đó ⇒ `best_ended` chỉ tăng, đúng nghĩa "max của các sự kiện đã kết thúc".

**Độ phức tạp Cách 3:** `O(n log n)`.

### 7.3 Cách 4 — Quét điểm sự kiện (sweep line) không cần heap

Tạo các điểm `(start, 1, value)` (mở) và `(end + 1, 0, value)` (đóng — kết thúc **trước** thời điểm `end + 1`). Sắp theo thời điểm, **đóng trước mở** khi cùng thời điểm. Khi gặp điểm đóng: cập nhật `best_ended`; điểm mở: `best = max(best, value + best_ended)`.

In [10]:
class Solution:
    def maxTwoEvents(self, events: List[List[int]]) -> int:
        points = []
        for s, e, v in events:
            points.append((s, 1, v))                  # mở
            points.append((e + 1, 0, v))              # đóng (0 < 1: đóng xử lý trước khi cùng thời điểm)
        points.sort()
        best_ended = best = 0
        for _, kind, v in points:
            if kind == 0:
                best_ended = max(best_ended, v)
            else:
                best = max(best, v + best_ended)
        return best

print([Solution().maxTwoEvents(ev) for ev, _ in examples])

[4, 5, 8]


**Độ phức tạp Cách 4:** `O(n log n)` (sắp `2n` điểm).

### 7.4 Cách 5 — Đối xứng: sắp theo **end** + max tiền tố + `bisect` trên `end`

Cố định sự kiện **sau** `f`; các sự kiện kết thúc trước `f.start` (tức `end ≤ f.start − 1`) là một **tiền tố** khi sắp theo `end`. `pre[i]` = max value của `i` sự kiện đầu (theo end). Ứng viên của `f` là `f.value + pre[bisect_right(ends, f.start − 1)]`.

In [11]:
def max_two_prefix_by_end(events):
    ev = sorted(events, key=lambda x: x[1])
    ends = [e[1] for e in ev]; n = len(ev)
    pre = [0] * (n + 1)
    for i in range(n):
        pre[i + 1] = max(pre[i], ev[i][2])
    return max(v + pre[bisect.bisect_right(ends, s - 1)] for s, e, v in ev)

print([max_two_prefix_by_end(ev) for ev, _ in examples])
rng = random.Random(74)
for _ in range(1000):
    ev = [[s, s + rng.randint(0, 4), rng.randint(1, 9)] for s in (rng.randint(1, 8) for _ in range(rng.randint(2, 8)))]
    assert max_two_prefix_by_end(ev) == max_two_bruteforce(ev)
print("Cách 5 khớp vét cạn trên 1000 bộ")

[4, 5, 8]
Cách 5 khớp vét cạn trên 1000 bộ


Cách 2 và Cách 5 là hai mặt của cùng một ý: "cố định một sự kiện, lấy max của nửa còn lại đã sắp". Cách 3 (heap) chính là Cách 5 làm **trực tuyến** khi quét theo `start`.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 1** `[[1,3,2],[4,5,2],[2,4,3]]`:

In [12]:
def trace(events):
    pts = sorted([(s, 1, v) for s, e, v in events] + [(e + 1, 0, v) for s, e, v in events])
    be = best = 0
    for t, kind, v in pts:
        if kind == 0:
            be = max(be, v); print(f"t={t}: đóng (giá trị {v}) -> best_ended={be}")
        else:
            best = max(best, v + be); print(f"t={t}: mở   (giá trị {v}) -> ứng viên {v} + {be} = {v + be}, best={best}")
    return best

assert trace(examples[0][0]) == 4

t=1: mở   (giá trị 2) -> ứng viên 2 + 0 = 2, best=2
t=2: mở   (giá trị 3) -> ứng viên 3 + 0 = 3, best=3
t=4: đóng (giá trị 2) -> best_ended=2
t=4: mở   (giá trị 2) -> ứng viên 2 + 2 = 4, best=4
t=5: đóng (giá trị 3) -> best_ended=3
t=6: đóng (giá trị 2) -> best_ended=3


| t | loại | giá trị | best_ended | best |
|---|---|---|---|---|
| 1 | mở | 2 | 0 | 2 |
| 2 | mở | 3 | 0 | 3 |
| 4 | **đóng** (`[1,3]` kết thúc trước 4) | 2 | 2 | 3 |
| 4 | mở `[4,5]` | 2 | 2 | **4** |
| 5 | đóng (`[2,4]`) | 3 | 3 | 4 |
| 6 | đóng (`[4,5]`) | 2 | 3 | 4 |

Ở `t = 4`, điểm **đóng** của `[1,3]` được xử lý **trước** điểm mở của `[4,5]` nhờ khóa `0 < 1` — đúng vì `[1,3]` và `[4,5]` không giao nhau.

Trace thêm Cách 3 (heap) trên **Ví dụ 3** để thấy `best_ended` chỉ tăng khi sự kiện thực sự đã kết thúc:

In [13]:
def trace_heap(events):
    ev = sorted(events); heap = []; be = best = 0
    for s, e, v in ev:
        popped = []
        while heap and heap[0][0] < s:
            x = heapq.heappop(heap); popped.append(x); be = max(be, x[1])
        best = max(best, v + be)
        print(f"xét [{s},{e},{v}]: lấy ra {popped} -> best_ended={be}, ứng viên {v + be}, best={best}")
        heapq.heappush(heap, (e, v))
    return best

assert trace_heap(examples[2][0]) == 8

xét [1,5,1]: lấy ra [] -> best_ended=0, ứng viên 1, best=1
xét [1,5,3]: lấy ra [] -> best_ended=0, ứng viên 3, best=3
xét [6,6,5]: lấy ra [(5, 1), (5, 3)] -> best_ended=3, ứng viên 8, best=8


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — sắp theo start, max hậu tố, `bisect`.

In [14]:
class Solution:
    def maxTwoEvents(self, events: List[List[int]]) -> int:
        events.sort()
        starts = [e[0] for e in events]
        n = len(events)
        suf = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            suf[i] = max(suf[i + 1], events[i][2])
        best = 0
        for s, e, v in events:
            j = bisect.bisect_left(starts, e + 1)
            best = max(best, v + suf[j])
        return best

for ev, exp in examples:
    got = Solution().maxTwoEvents([x[:] for x in ev]); print(got, exp); assert got == exp

4 4
5 5
8 8


## 10. Vì sao thuật toán đúng

**Phân loại lời giải.** Một lời giải tối ưu là một sự kiện `{x}` hoặc hai sự kiện không giao nhau; trong trường hợp hai, gọi `e` là sự kiện kết thúc sớm hơn (khi đó `f.start ≥ e.end + 1`).

**Cận dưới:** với mỗi `e`, `v_e + suf[j]` (với `j = bisect_left(starts, e.end + 1)`) là tổng của `e` và một sự kiện **có thật** bắt đầu từ `e.end + 1` (hoặc chỉ `e` nếu `suf[j] = 0`) — không giao với `e` (sự kiện đó bắt đầu sau khi `e` kết thúc). Nên mọi giá trị ta tính đều đạt được.

**Cận trên:** lời giải `{e, f}` có `f.start ≥ e.end + 1` ⇒ `f` nằm trong hậu tố `j` ⇒ `v_f ≤ suf[j]` ⇒ `v_e + v_f ≤` ứng viên của `e`. Lời giải `{x}` ≤ ứng viên của `x` (vì `suf ≥ 0`). Vậy `max` ứng viên = tối ưu. ∎

**Cách 4.** Điểm đóng `(end + 1, 0)` đứng trước điểm mở `(start, 1)` khi `start ≥ end + 1`, kể cả khi bằng nhau ⇒ khi mở `f`, `best_ended` gồm đúng các `e` với `e.end + 1 ≤ f.start`.

### 10.1 Kiểm chứng thực nghiệm

So bốn cách với vét cạn trên dữ liệu có **nhiều biên chạm nhau** (thời gian trong miền nhỏ), để bắt lỗi `<` / `≤`.

In [15]:
def rand_events(rng, n, T, V):
    out = []
    for _ in range(n):
        s = rng.randint(1, T); e = rng.randint(s, min(T, s + rng.randint(0, T)))
        out.append([s, e, rng.randint(1, V)])
    return out

rng = random.Random(1010)
for _ in range(2000):
    ev = rand_events(rng, rng.randint(2, 9), rng.choice([3, 6, 20]), rng.choice([3, 10]))
    ref = max_two_bruteforce(ev)
    assert ref == max_two_suffix(ev) == max_two_heap(ev) == Solution().maxTwoEvents([x[:] for x in ev])
print("4 cách = vét cạn trên 2000 bộ sự kiện (thời gian nhỏ, nhiều biên chạm)")

4 cách = vét cạn trên 2000 bộ sự kiện (thời gian nhỏ, nhiều biên chạm)


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n²)` | `O(1)` |
| 2. Hậu tố + `bisect` | `O(n log n)` | `O(n)` |
| 3. Heap | `O(n log n)` | `O(n)` |
| 4. Sweep line | `O(n log n)` | `O(n)` |

In [16]:
rng = random.Random(2054)
big = rand_events(rng, 10 ** 5, 10 ** 9, 10 ** 6)
for name, f in [("suffix", max_two_suffix), ("heap", max_two_heap), ("submit", Solution().maxTwoEvents)]:
    t = time.perf_counter(); r = f([x[:] for x in big]); print(f"{name:<7}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
small = big[:2000]
t = time.perf_counter(); max_two_bruteforce(small); print(f"brute n = 2000: {(time.perf_counter() - t) * 1000:.0f} ms")

suffix : 1999964 (204 ms)
heap   : 1999964 (200 ms)


submit : 1999964 (157 ms)
brute n = 2000: 179 ms


## 12. Bộ test

In [17]:
ALL = {"suffix": max_two_suffix, "heap": max_two_heap, "submit": Solution().maxTwoEvents}
def run_all(ev, brute=True):
    res = {k: f([x[:] for x in ev]) for k, f in ALL.items()}
    if brute:
        res["brute"] = max_two_bruteforce(ev)
    assert len(set(res.values())) == 1, (ev, res)
    return res["submit"]

for ev, exp in examples:
    assert run_all(ev) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [([[1, 3, 2], [3, 4, 3]], 3), ([[1, 3, 2], [4, 4, 3]], 5), ([[1, 1, 1], [1, 1, 1]], 1), ([[1, 1, 1], [2, 2, 1]], 2),
        ([[1, 10 ** 9, 10 ** 6], [1, 1, 1]], 10 ** 6), ([[5, 6, 1], [1, 4, 1], [7, 8, 10]], 11)]
for ev, exp in edge:
    assert run_all(ev) == exp, (ev, run_all(ev))
rng = random.Random(20540)
for _ in range(400):
    run_all(rand_events(rng, rng.randint(2, 12), rng.choice([5, 30, 10 ** 9]), rng.choice([5, 10 ** 6])))
for _ in range(3):
    run_all(rand_events(rng, 3000, 10 ** 4, 100), brute=False)
print("Ví dụ, 6 edge case, 400 bộ nhỏ (vét cạn), 3 bộ 3000 sự kiện: OK")

Ví dụ, 6 edge case, 400 bộ nhỏ (vét cạn), 3 bộ 3000 sự kiện: OK


### 12.1 Kiểm tra tính chất

- Hoán vị `events` không đổi đáp án; đáp án `≥ max value` và `≤ tổng hai giá trị lớn nhất`.
- Kéo dài một sự kiện (giảm start hoặc tăng end) không làm đáp án tăng.
- Thêm một sự kiện không làm đáp án giảm.

In [18]:
rng = random.Random(121)
f = Solution().maxTwoEvents
for _ in range(500):
    ev = rand_events(rng, rng.randint(2, 10), 15, 10); a = f([x[:] for x in ev])
    sh = [x[:] for x in ev]; rng.shuffle(sh); assert f(sh) == a
    vals = sorted(x[2] for x in ev)
    assert vals[-1] <= a <= vals[-1] + vals[-2]
    i = rng.randrange(len(ev)); g = [x[:] for x in ev]; g[i][0] = max(1, g[i][0] - 1); g[i][1] += 1
    assert f(g) <= a
    assert f([x[:] for x in ev] + [rand_events(rng, 1, 15, 10)[0]]) >= a
print("3 tính chất đúng trên 500 bộ")

3 tính chất đúng trên 500 bộ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi `end == start` là không giao nhau (quên biên bao gồm)

In [19]:
def wrong_inclusive(events):
    ev = sorted(events); starts = [e[0] for e in ev]; n = len(ev)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1): suf[i] = max(suf[i + 1], ev[i][2])
    return max(v + suf[bisect.bisect_left(starts, e)] for s, e, v in ev)     # SAI: phải e + 1

ev = [[1, 3, 2], [3, 4, 3]]        # tự dựng (đối chiếu lại đề gốc): chung thời điểm 3
print("sai :", wrong_inclusive(ev), "| đúng:", Solution().maxTwoEvents([x[:] for x in ev]), "| brute:", max_two_bruteforce(ev))

sai : 5 | đúng: 3 | brute: 3


**Vì sao sai:** `bisect_left(starts, e)` lấy cả sự kiện bắt đầu **đúng** lúc `e` kết thúc — đề nói thời gian bao gồm hai đầu nên chúng giao nhau. Thậm chí có thể ghép `e` với **chính nó** (`start = end`). **Sửa:** tìm `start ≥ end + 1`.

### Lỗi 2 — Bắt buộc chọn đúng hai sự kiện

In [20]:
def wrong_exactly_two(events):
    n = len(events); best = 0
    for i in range(n):
        for j in range(i + 1, n):
            if disjoint(events[i], events[j]): best = max(best, events[i][2] + events[j][2])
    return best                                                   # SAI: thiếu trường hợp một sự kiện

for ev, exp in examples:
    print("sai :", wrong_exactly_two(ev), "| đúng:", exp)

sai : 4 | đúng: 4
sai : 4 | đúng: 5
sai : 8 | đúng: 8


**Vì sao sai:** "**tối đa** hai" — chọn một sự kiện cũng hợp lệ. Ví dụ 2: sự kiện `[1,5,5]` giao với mọi sự kiện khác nhưng tốt hơn mọi cặp. **Sửa:** khởi tạo `best = max value` (hoặc dùng `suf[n] = 0`).

### Lỗi 3 — Tham lam lấy hai giá trị lớn nhất (bỏ qua giao nhau)

In [21]:
def wrong_top2(events):
    vals = sorted((v for _, _, v in events), reverse=True)
    return vals[0] + vals[1]                                     # SAI

for ev, exp in examples:
    print("sai :", wrong_top2(ev), "| đúng:", exp)

sai : 5 | đúng: 4
sai : 7 | đúng: 5
sai : 8 | đúng: 8


**Vì sao sai:** hai sự kiện giá trị cao nhất có thể chồng nhau. **Sửa:** kiểm tra ràng buộc thời gian.

### Lỗi 4 — Dùng max **tiền tố** thay vì hậu tố

In [22]:
def wrong_prefix(events):
    ev = sorted(events); starts = [e[0] for e in ev]; n = len(ev)
    pre = [0] * (n + 1)
    for i in range(n): pre[i + 1] = max(pre[i], ev[i][2])
    best = 0
    for s, e, v in ev:
        j = bisect.bisect_left(starts, e + 1)
        best = max(best, v + (pre[n] - 0 if j < n else 0))      # SAI: lấy max toàn bộ thay vì từ j
    return best

for ev, exp in examples:
    print("sai :", wrong_prefix(ev), "| đúng:", exp)

sai : 5 | đúng: 4
sai : 7 | đúng: 5
sai : 8 | đúng: 8


**Vì sao sai:** cần max của các sự kiện **bắt đầu sau** `e` (hậu tố từ `j`), không phải của toàn bộ hay phần đầu. Ở Ví dụ 1, `pre[n] = 3` lấy cả sự kiện `[2,4,3]` (giao với `[1,3]`). **Sửa:** `suf[j]`.

### Lỗi 5 — Sweep line đặt **mở trước đóng** khi cùng thời điểm

In [23]:
def wrong_sweep_order(events):
    pts = sorted([(s, 0, v) for s, e, v in events] + [(e + 1, 1, v) for s, e, v in events])   # SAI: mở (0) trước đóng (1)
    be = best = 0
    for _, kind, v in pts:
        if kind == 1: be = max(be, v)
        else: best = max(best, v + be)
    return best

for ev, exp in examples:
    print("sai :", wrong_sweep_order(ev), "| đúng:", exp)

sai : 3 | đúng: 4
sai : 5 | đúng: 5
sai : 5 | đúng: 8


**Vì sao sai:** ở `t = 4` của Ví dụ 1, `[4,5]` mở **trước** khi `[1,3]` được tính là đã kết thúc ⇒ bỏ lỡ cặp hợp lệ. **Sửa:** đóng (khóa nhỏ hơn) trước mở.

### Lỗi 6 — Vét cạn cặp với dữ liệu lớn

In [24]:
for n in (500, 1000, 2000):
    ev = rand_events(random.Random(n), n, 10 ** 9, 10 ** 6)
    t = time.perf_counter(); max_two_bruteforce(ev); tb = time.perf_counter() - t
    t = time.perf_counter(); Solution().maxTwoEvents([x[:] for x in ev]); ts = time.perf_counter() - t
    print(f"n={n:>5}: vét cạn {tb * 1000:6.0f} ms | hậu tố {ts * 1000:.1f} ms")

n=  500: vét cạn     12 ms | hậu tố 0.3 ms
n= 1000: vét cạn     52 ms | hậu tố 0.8 ms


n= 2000: vét cạn    192 ms | hậu tố 1.3 ms


**Vì sao không tốt:** thời gian tăng gấp 4 khi `n` gấp đôi; `n = 10⁵` cần ~hàng giờ. **Sửa:** `O(n log n)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn hai đối tượng thỏa ràng buộc thứ tự, tối đa tổng" ⇒ **cố định một**, truy vấn "tốt nhất trong phần còn lại" bằng max tiền tố/hậu tố + `bisect`, hoặc quét + heap.
- Khoảng **đóng** trên số nguyên: "không giao" ⇔ `next.start ≥ prev.end + 1`.
- Sweep line: chọn thứ tự xử lý điểm cùng thời điểm **cẩn thận**.
- "Tối đa `k`" luôn gồm cả ít hơn `k` (ở đây một sự kiện).
- Tổng quát `k` sự kiện ⇒ DP **1751 Maximum Number of Events That Can Be Attended II**.

## 15. Biến thể của bài

1. **1751**: tối đa `k` sự kiện — DP `dp[i][t]` + `bisect`, `O(nk log n)` (Bài tập 5).
2. **1235 Maximum Profit in Job Scheduling**: không giới hạn số sự kiện, khoảng **nửa mở** (`end ≤ next.start`).
3. **435 Non-overlapping Intervals**: tối đa **số lượng** khoảng không giao — tham lam theo `end`.
4. **Ba sự kiện**: thêm max tiền tố cho sự kiện đầu và max hậu tố cho sự kiện cuối, cố định sự kiện giữa.
5. **Khoảng nửa mở** `[start, end)`: đổi `e + 1` thành `e`.

### 15.1 Code biến thể: 1235 Job Scheduling (khoảng nửa mở, không giới hạn số lượng)

In [25]:
def job_scheduling(startTime, endTime, profit):
    jobs = sorted(zip(endTime, startTime, profit))
    ends = [j[0] for j in jobs]; dp = [0] * (len(jobs) + 1)
    for i, (e, s, p) in enumerate(jobs):
        k = bisect.bisect_right(ends, s, 0, i)            # job kết thúc <= s
        dp[i + 1] = max(dp[i], dp[k] + p)
    return dp[-1]

def job_brute(st, en, pr):
    n = len(st); best = 0
    for m in range(1 << n):
        sel = [i for i in range(n) if m >> i & 1]
        sel.sort(key=lambda i: st[i])
        if all(en[sel[t]] <= st[sel[t + 1]] for t in range(len(sel) - 1)):
            best = max(best, sum(pr[i] for i in sel))
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 8); st = [rng.randint(1, 10) for _ in range(n)]; en = [s + rng.randint(1, 5) for s in st]; pr = [rng.randint(1, 9) for _ in range(n)]
    assert job_scheduling(st, en, pr) == job_brute(st, en, pr)
print("1235 khớp vét cạn;", job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]), "(LeetCode: 120)")

1235 khớp vét cạn; 120 (LeetCode: 120)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | Vét cạn | `O(n²)` | `O(1)` | Kiểm chứng |
| 2. Hậu tố + bisect | Cố định sự kiện trước | `O(n log n)` | `O(n)` | **Bản nộp** |
| 3. Heap | Cố định sự kiện sau | `O(n log n)` | `O(n)` | Mở rộng sang dạng "đã kết thúc" |
| 4. Sweep line | Điểm mở/đóng | `O(n log n)` | `O(n)` | Gọn, không bisect |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[1,1,1],[1,1,1]]`; (b) `[[1,2,4],[3,4,3],[2,3,10]]`; (c) `[[1,5,1],[6,10,1],[11,12,1]]`; (d) `[[1,3,5],[3,5,5]]`.

**Bài 2 (tính tay):** Với `[[5,6,1],[1,4,1],[7,8,10]]`, lập bảng `starts`, `suf`, và ứng viên của từng sự kiện.

**Bài 3 (điền TODO):** Hoàn thiện `suffix_max(vals)`.

**Bài 4 (tự viết lại):** Viết `max_three_events(events)` — tối đa **ba** sự kiện không giao nhau; đối chiếu vét cạn.

**Bài 5 (mini challenge):** Viết `max_k_events(events, k)` (LeetCode 1751) bằng DP `dp[i][t]` = tổng lớn nhất dùng tối đa `t` sự kiện trong các sự kiện từ chỉ số `i` (đã sắp theo start); đối chiếu vét cạn với `n ≤ 8`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def suffix_max(vals):
    suf = [0] * (len(vals) + 1)
    # TODO: duyệt i từ cuối về đầu: suf[i] = max(suf[i + 1], vals[i])
    return suf

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_three_events(events):
    # TODO: dùng max_k_events với k = 3, hoặc cố định sự kiện giữa + max tiền tố (theo end) + max hậu tố (theo start)
    pass

def max_k_events(events, k):
    # TODO: sắp theo start; nxt[i] = bisect_left(starts, end_i + 1); dp[i][t] = max(dp[i+1][t], v_i + dp[nxt[i]][t-1])
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: kiểm tra giao nhau với biên bao gồm. Mức 2: nhớ trường hợp một sự kiện. Mức 3: (a) `1`; (b) `10` (`[2,3,10]` giao cả hai; cặp `[1,2]`+`[3,4]` chỉ được 7); (c) `2`; (d) `5` (chung thời điểm 3).

**Bài 2.** Mức 1: sắp: `[1,4,1], [5,6,1], [7,8,10]`, `starts = [1,5,7]`. Mức 2: `suf = [10, 10, 10, 0]`. Mức 3: ứng viên `1 + suf[1] = 11`, `1 + suf[2] = 11`, `10 + suf[3] = 10` ⇒ `11`.

**Bài 3.** Mức 1: mảng dài `n + 1`, `suf[n] = 0`. Mức 2: duyệt ngược. Mức 3: `max` hai giá trị.

**Bài 4.** Mức 1: tổng quát hóa thành DP `k`. Mức 2: `k = 3`. Mức 3: vét cạn mọi bộ ba để kiểm.

**Bài 5.** Mức 1: `nxt[i]`: sự kiện đầu tiên bắt đầu sau khi `i` kết thúc. Mức 2: chọn hoặc bỏ `i`. Mức 3: `dp[n][t] = 0`, `dp[i][0] = 0`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2, 3
f = Solution().maxTwoEvents
b1 = [f(x) for x in ([[1, 1, 1], [1, 1, 1]], [[1, 2, 4], [3, 4, 3], [2, 3, 10]], [[1, 5, 1], [6, 10, 1], [11, 12, 1]], [[1, 3, 5], [3, 5, 5]])]
print("Bài 1:", b1); assert b1 == [1, 10, 2, 5]
ev = sorted([[5, 6, 1], [1, 4, 1], [7, 8, 10]]); st = [e[0] for e in ev]
def suffix_max(vals):
    suf = [0] * (len(vals) + 1)
    for i in range(len(vals) - 1, -1, -1):
        suf[i] = max(suf[i + 1], vals[i])
    return suf
suf = suffix_max([e[2] for e in ev])
cands = [v + suf[bisect.bisect_left(st, e + 1)] for s, e, v in ev]
print("Bài 2: starts", st, "suf", suf, "ứng viên", cands); assert suf == [10, 10, 10, 0] and max(cands) == 11
print("Bài 3: OK")

Bài 1: [1, 10, 2, 5]
Bài 2: starts [1, 5, 7] suf [10, 10, 10, 0] ứng viên [11, 11, 10]
Bài 3: OK


In [29]:
# Bài 4, 5
def max_k_events(events, k):
    ev = sorted(events); n = len(ev); st = [e[0] for e in ev]
    nxt = [bisect.bisect_left(st, e[1] + 1) for e in ev]
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        for t in range(1, k + 1):
            dp[i][t] = max(dp[i + 1][t], ev[i][2] + dp[nxt[i]][t - 1])
    return dp[0][k]

def max_three_events(events):
    return max_k_events(events, 3)

def k_brute(events, k):
    n = len(events); best = 0
    for m in range(1, 1 << n):
        sel = [events[i] for i in range(n) if m >> i & 1]
        if len(sel) > k: continue
        if all(disjoint(a, b) for i, a in enumerate(sel) for b in sel[i + 1:]):
            best = max(best, sum(x[2] for x in sel))
    return best

rng = random.Random(5)
for _ in range(300):
    ev = rand_events(rng, rng.randint(1, 8), 12, 9); k = rng.randint(1, 4)
    assert max_k_events(ev, k) == k_brute(ev, k)
    if len(ev) >= 2: assert max_k_events(ev, 2) == Solution().maxTwoEvents([x[:] for x in ev])
print("Bài 4, 5: OK |", max_three_events([[1, 2, 4], [3, 4, 3], [2, 3, 1], [5, 6, 2]]))

Bài 4, 5: OK | 9


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
events.sort()                                  # theo start
starts = [s for s, _, _ in events]
suf = [0] * (n + 1)
for i in range(n - 1, -1, -1): suf[i] = max(suf[i + 1], events[i][2])
best = 0
for s, e, v in events:
    j = bisect_left(starts, e + 1)             # bắt đầu SAU khi e kết thúc (biên bao gồm)
    best = max(best, v + suf[j])               # suf[n] = 0 -> một sự kiện
return best
```

**Checklist:**
- [ ] Biên bao gồm: tìm `start ≥ end + 1`?
- [ ] Có xét trường hợp **một** sự kiện?
- [ ] Dùng max **hậu tố** từ vị trí `bisect`?
- [ ] Sweep line: đóng trước mở khi cùng thời điểm?
- [ ] `O(n log n)`, không vét cạn cặp?